# Project: Weird digit detector (autoencoder) 🔍
Train an **autoencoder** only on normal handwritten digits: squeeze 64 pixels through an 8-number bottleneck and reconstruct them. Anything it reconstructs badly is "weird". Test it on digits with corruptions it has never seen: noise, inversion, scribbles. Unsupervised anomaly detection with a neural net.

Topic: [[Neural Networks]], [[Anomaly Detection]], [[Dimensionality Reduction]] · guide note: [[Project - Weird Digit Detector]]

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import numpy as np, matplotlib.pyplot as plt, torch, torch.nn as nn, torch.nn.functional as F
from sklearn.datasets import load_digits
from sklearn.metrics import roc_auc_score
torch.manual_seed(0)
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing
X, y = load_digits(return_X_y=True); X = (X / 16).astype(np.float32)
rng = np.random.default_rng(0); perm = rng.permutation(len(X)); tr, te = perm[:1300], perm[1300:]
Xtr = torch.tensor(X[tr]); Xte = X[te].copy()
weird = Xte.copy(); kind = np.array(["normal"] * len(te), dtype=object)
idx = rng.choice(len(te), 150, replace=False)
for j, i in enumerate(idx):
    if j % 3 == 0: weird[i] = np.clip(weird[i] + rng.normal(0, 0.5, 64), 0, 1); kind[i] = "noise"
    elif j % 3 == 1: weird[i] = 1 - weird[i]; kind[i] = "inverted"
    else: s = weird[i].reshape(8, 8); s[rng.integers(8), :] = 1; s[:, rng.integers(8)] = 1; kind[i] = "scribble"
is_weird = kind != "normal"
print(f"train {len(tr)} normal digits; test {len(te)} digits of which {is_weird.sum()} are corrupted")

## 1. The autoencoder
Encoder: `Linear(64→32)` → ReLU → `Linear(32→k)`. Decoder: `Linear(k→32)` → ReLU → `Linear(32→64)` → **Sigmoid** (pixels are in [0, 1]).

In [ ]:
class AE(nn.Module):
    def __init__(self, k=8):
        super().__init__()
        # TODO 1: self.enc and self.dec as described
        raise NotImplementedError  # TODO 1
    def forward(self, x): return self.dec(self.enc(x))

In [ ]:
def _t_ae():
    m = AE(8); out = m(torch.rand(5, 64))
    return out.shape == (5, 64) and float(out.min()) >= 0 and float(out.max()) <= 1 and m.enc(torch.rand(2, 64)).shape == (2, 8)
check("autoencoder", _t_ae, "Output must be (batch, 64) in [0, 1]; the code must be (batch, k).")
if ready("autoencoder"):
    model = AE(8); opt = torch.optim.Adam(model.parameters(), 3e-3)
    for epoch in range(300):
        for b_ in torch.randperm(len(Xtr)).split(64):
            l = F.mse_loss(model(Xtr[b_]), Xtr[b_]); opt.zero_grad(); l.backward(); opt.step()
    print(f"final training MSE {l.item():.4f}")

## 2. Anomaly score
Return the **per-example mean squared reconstruction error** as a NumPy array (no gradients).

In [ ]:
def recon_error(model, X):
    # TODO 2: mean over pixels of (model(X) − X)², one number per row
    raise NotImplementedError  # TODO 2

In [ ]:
check("recon error", lambda: recon_error(nn.Identity(), torch.rand(4, 64)).shape == (4,) and np.allclose(recon_error(nn.Identity(), torch.rand(4, 64)), 0),
      "Identity 'model' → zero error; one score per example.")
if ready("autoencoder", "recon error"):
    score = recon_error(model, torch.tensor(weird))
    auc = roc_auc_score(is_weird, score); print(f"ROC AUC (weird vs normal): {auc:.3f}")
    for k_ in ("noise", "inverted", "scribble"): print(f"  {k_:<9} AUC {roc_auc_score(is_weird[(kind == k_) | ~is_weird], score[(kind == k_) | ~is_weird]):.3f}")
    top = np.argsort(-score)[:8]
    fig, ax = plt.subplots(2, 8, figsize=(12, 3.2))
    with torch.no_grad(): rec = model(torch.tensor(weird[top])).numpy()
    for j, i in enumerate(top):
        ax[0, j].imshow(weird[i].reshape(8, 8), cmap="gray_r"); ax[0, j].set_title(kind[i], fontsize=8); ax[1, j].imshow(rec[j].reshape(8, 8), cmap="gray_r")
        ax[0, j].axis("off"); ax[1, j].axis("off")
    plt.suptitle("weirdest test images (top) and their reconstructions (bottom)"); plt.show()
    check("finds weird digits", lambda: auc > 0.9, "Reconstruction error should separate corrupted from normal digits with AUC > 0.9.")

<details><summary>▶ Solution</summary>

```python
        # inside AE.__init__
        self.enc = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, k))
        self.dec = nn.Sequential(nn.Linear(k, 32), nn.ReLU(), nn.Linear(32, 64), nn.Sigmoid())
```

```python
def recon_error(model, X):
    with torch.no_grad():
        return ((model(X) - X) ** 2).mean(1).numpy()
```
</details>

## Stretch goals
- Bottleneck size: k = 2, 8, 32. Too small reconstructs nothing well; too big reconstructs *everything* well, including the weird ones. Plot AUC vs. k.
- Compare with PCA reconstruction error at the same k ([[PCA]]): a linear autoencoder.
- Train only on digits 0–8 and see if the autoencoder flags 9s as weird (novelty detection).